In [14]:
# Fine-tuning Qwen2-0.5B on tiny-codes dataset
# Using 2025 best practices with SFTTrainer from TRL library
# Reference: https://www.philschmid.de/fine-tune-llms-in-2025

import torch
from transformers import (
    AutoModelForCausalLM, 
    AutoTokenizer, 
    TrainingArguments,
    BitsAndBytesConfig
)
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
import os

print("Library versions:")
print(f"  PyTorch: {torch.__version__}")
print(f"  Transformers: {__import__('transformers').__version__}")
print(f"  TRL: {__import__('trl').__version__}")
print(f"  Datasets: {__import__('datasets').__version__}")
print(f"  PEFT: {__import__('peft').__version__}")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Library versions:
  PyTorch: 2.5.1+cu121
  Transformers: 4.57.1
  TRL: 0.24.0
  Datasets: 4.2.0
  PEFT: 0.17.1

CUDA available: True
GPU: NVIDIA GeForce RTX 5090
GPU memory: 33.66 GB


In [15]:
# Configuration - 2025 Best Practices
MODEL_NAME = "Qwen/Qwen2-0.5B"
DATASET_NAME = "nampdn-ai/tiny-codes"
OUTPUT_DIR = "./qwen2-tiny-codes-lora"

# Training hyperparameters
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4  # Standard for LoRA/QLoRA
NUM_EPOCHS = 1
MAX_STEPS = -1  # -1 means train for full epochs

# QLoRA configuration (4-bit quantization)
# NOTE: Disabled due to RTX 5090 compatibility issues with bitsandbytes
# The RTX 5090 (sm_120) is not yet supported by PyTorch 2.5.1 compiled kernels
USE_QLORA = False
LOAD_IN_4BIT = False

# LoRA configuration
LORA_R = 16  # LoRA attention dimension
LORA_ALPHA = 32  # Alpha parameter for LoRA scaling
LORA_DROPOUT = 0.05

# Dataset size (use smaller subset for testing)
DATASET_SIZE = 10000  # Set to None for full dataset

print("Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Using QLoRA: {USE_QLORA}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  LoRA r={LORA_R}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"\nNote: QLoRA disabled - using LoRA with bfloat16 instead")
print(f"      Your RTX 5090 will still train efficiently with 33GB VRAM")

Configuration:
  Model: Qwen/Qwen2-0.5B
  Dataset: nampdn-ai/tiny-codes
  Using QLoRA: False
  Max sequence length: 1024
  Batch size: 2
  Gradient accumulation: 4
  Effective batch size: 8
  Learning rate: 0.0002
  LoRA r=16, alpha=32, dropout=0.05

Note: QLoRA disabled - using LoRA with bfloat16 instead
      Your RTX 5090 will still train efficiently with 33GB VRAM


In [16]:
# Load tokenizer
print(f"\nLoading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ensure tokenizer has pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Tokenizer loaded. Vocab size: {len(tokenizer)}")
print(f"  EOS token: {tokenizer.eos_token} (ID: {tokenizer.eos_token_id})")
print(f"  PAD token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")


Loading tokenizer: Qwen/Qwen2-0.5B
Tokenizer loaded. Vocab size: 151646
  EOS token: <|endoftext|> (ID: 151643)
  PAD token: <|endoftext|> (ID: 151643)


In [17]:
# Load model with QLoRA (4-bit quantization)
print(f"\nLoading model: {MODEL_NAME}")

if USE_QLORA:
    # Configure 4-bit quantization for QLoRA
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",  # Normal Float 4-bit
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,  # Nested quantization for additional memory savings
    )
    print("Using 4-bit quantization (QLoRA)")
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        # Flash Attention 2 disabled - not required for this small model
    )
    
    # Prepare model for k-bit training
    model = prepare_model_for_kbit_training(model)
    
else:
    # Standard bfloat16 loading
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map="auto",
    )

print(f"Model loaded successfully")
print(f"  Total parameters: {model.num_parameters() / 1e6:.2f}M")
print(f"  Device: {model.device}")

`torch_dtype` is deprecated! Use `dtype` instead!



Loading model: Qwen/Qwen2-0.5B
Model loaded successfully
  Total parameters: 494.03M
  Device: cuda:0


In [18]:
# Configure LoRA adapters
print("\nConfiguring LoRA...")

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],  # Qwen2 attention modules
)

# Apply LoRA to the model
model = get_peft_model(model, peft_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())

print(f"LoRA configured successfully")
print(f"  Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")
print(f"  Total parameters: {all_params:,}")


Configuring LoRA...


RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Load dataset using streaming (no caching)
print(f"\nLoading dataset: {DATASET_NAME}")
print("Using streaming mode (no caching)...")

# Stream dataset
dataset_stream = load_dataset(DATASET_NAME, split="train", streaming=True)

# Take subset if specified
if DATASET_SIZE:
    print(f"Loading first {DATASET_SIZE:,} examples...")
    examples = []
    for i, example in enumerate(dataset_stream):
        if i >= DATASET_SIZE:
            break
        examples.append(example)
    
    from datasets import Dataset
    dataset = Dataset.from_list(examples)
else:
    # Load full dataset (this will take longer)
    print("Loading full dataset...")
    dataset = list(dataset_stream)
    from datasets import Dataset
    dataset = Dataset.from_list(dataset)

print(f"Loaded {len(dataset):,} examples")
print(f"Dataset columns: {dataset.column_names}")

# Format dataset for chat template
def format_chat(example):
    """Format examples using chat template"""
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["response"]}
    ]
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return example

# Apply formatting
print("Formatting dataset with chat template...")
dataset = dataset.map(format_chat, num_proc=4)

# Split into train/validation
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training examples: {len(train_dataset):,}")
print(f"Validation examples: {len(eval_dataset):,}")
print(f"\nExample formatted text:")
print(train_dataset[0]["text"][:200] + "...")

In [ ]:
# Configure SFTTrainer - 2025 Best Practices
print("\nConfiguring SFTTrainer...")

# Use SFTConfig (new in 2025, replaces TrainingArguments for SFT)
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Training parameters
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    # Optimization
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    optim="paged_adamw_8bit" if USE_QLORA else "adamw_torch",  # Memory-efficient optimizer for QLoRA
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    
    # Logging and evaluation
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    
    # Performance optimizations
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},  # Better memory efficiency
    
    # SFT-specific settings
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,  # Set to True for better GPU utilization (requires careful attention to masking)
    dataset_text_field="text",  # Field containing formatted text
    
    # Misc
    report_to=[],
    push_to_hub=False,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print("SFT Configuration:")
print(f"  Output: {OUTPUT_DIR}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Optimizer: {sft_config.optim}")
print(f"  Max sequence length: {MAX_SEQ_LENGTH}")
print(f"  Packing: {sft_config.packing}")

In [ ]:
# Initialize SFTTrainer
print("\nInitializing SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)

print("Trainer initialized successfully!")
print(f"  Total training steps: {len(train_dataset) // (BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS) * NUM_EPOCHS}")
print(f"  Warmup steps: {trainer.args.warmup_steps}")

In [ ]:
# Train the model
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

# Train
train_result = trainer.train()

print("\n" + "="*60)
print("TRAINING COMPLETED")
print("="*60)
print(f"\nFinal metrics:")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training time: {train_result.metrics.get('train_runtime', 0):.2f}s")

In [ ]:
# Save the model and adapters
print("\nSaving model...")

# Save LoRA adapters
trainer.save_model(OUTPUT_DIR)

# Save tokenizer
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✓ Model saved to: {OUTPUT_DIR}")
print(f"✓ LoRA adapters saved")
print(f"✓ Tokenizer saved")

print("\n" + "-"*60)
print("To merge LoRA weights into base model for inference:")
print("-"*60)
print("""
from peft import PeftModel
from transformers import AutoModelForCausalLM

# Load base model
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2-0.5B")

# Load and merge adapters
model = PeftModel.from_pretrained(base_model, "OUTPUT_DIR")
merged_model = model.merge_and_unload()

# Save merged model
merged_model.save_pretrained("./qwen2-tiny-codes-merged")
""")

In [ ]:
# Test the fine-tuned model
print("\n" + "="*60)
print("TESTING THE FINE-TUNED MODEL")
print("="*60 + "\n")

# Set model to eval mode
model.eval()

# Test prompt
test_prompt = "Write a Python function to calculate fibonacci numbers"

# Format with chat template
messages = [{"role": "user", "content": test_prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

print(f"Test prompt: {test_prompt}\n")
print("Generating response...")

# Generate
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "-"*60)
print("MODEL RESPONSE:")
print("-"*60)
print(response)
print("-"*60)

# Clean up GPU memory
torch.cuda.empty_cache()
print("\n✓ Test complete. GPU memory cleared.")